# 08 — Spend anomaly detector (Tier 3, fourth estimator)

Flags months whose spending departs materially from a venture's own
trailing pattern (standardised deviation against a rolling baseline).
Purpose is attribution, not classification: an anomaly alongside a hazard
increase indicates a discrete event; a hazard increase without one
indicates gradual deterioration.

Reads: `data/processed/synthetic_trajectories.csv`
Writes: `data/processed/spend_anomalies.csv`

In [1]:
import pandas as pd
import numpy as np

PROCESSED = "../data/processed"
traj = pd.read_csv(f"{PROCESSED}/synthetic_trajectories.csv")

ROLLING_WINDOW = 6
Z_THRESHOLD = 2.0

def flag_anomalies(g):
    g = g.sort_values("month_idx").copy()
    roll_mean = g["synthesized_spend"].rolling(ROLLING_WINDOW, min_periods=3).mean()
    roll_std = g["synthesized_spend"].rolling(ROLLING_WINDOW, min_periods=3).std()
    z = (g["synthesized_spend"] - roll_mean) / roll_std.replace(0, np.nan)
    g["z_score"] = z
    g["is_anomaly"] = z.abs() > Z_THRESHOLD
    return g

flagged = traj.groupby("object_id", group_keys=False).apply(flag_anomalies)
flagged.to_csv(f"{PROCESSED}/spend_anomalies.csv", index=False)

n_flagged = flagged["is_anomaly"].sum()
n_companies_with_anomaly = flagged.loc[flagged["is_anomaly"], "object_id"].nunique()
print(f"[done] flagged {n_flagged:,} anomalous months across {n_companies_with_anomaly:,} companies")
print(f"[done] ({100*n_companies_with_anomaly/flagged['object_id'].nunique():.1f}% of companies have at least one flagged month)")
print(f"[done] wrote {PROCESSED}/spend_anomalies.csv")

C:\Users\ranga\AppData\Local\Temp\ipykernel_29536\693332367.py:19: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  flagged = traj.groupby("object_id", group_keys=False).apply(flag_anomalies)


[done] flagged 867 anomalous months across 563 companies
[done] (29.7% of companies have at least one flagged month)
[done] wrote ../data/processed/spend_anomalies.csv
